# Lab 7: Word Sequences

In [ ]:
from pathlib import Path
import tarfile

paths = [Path('../sts/resources/test-gold.tgz'), Path('sts/resources/test-gold.tgz'), Path('test-gold.tgz')]
path = next((p for p in paths if p.is_file()), None)
if path is None:
    raise FileNotFoundError('Could not find test-gold.tgz')

with tarfile.open(path) as tar:
    pairs = [line.split('\t') for line in tar.extractfile('test-gold/STS.input.SMTeuroparl.txt').read().decode('utf-8').splitlines()]
    gs = [float(line) for line in tar.extractfile('test-gold/STS.gs.SMTeuroparl.txt').read().decode('utf-8').splitlines()]

assert len(pairs) == len(gs)
assert all(len(pair) == 2 for pair in pairs)

unique_sentences = list(dict.fromkeys(sentence for pair in pairs for sentence in pair))
print(f'{len(pairs)} pairs; {len(unique_sentences)} unique sentences')

459 pairs; 338 unique sentences


## Words and named entities

We use NLTK's named-entity chunker. We keep capital letters and punctuation for tagging, then lowercase the words and drop punctuation for Jaccard. The word baseline uses the same regex as in Labs 3 and 6. For words plus named entities, each detected entity counts as one item, such as `john smith`.

In [ ]:
import re

import nltk
from nltk.tree import Tree

resources = {
    'averaged_perceptron_tagger_eng': 'taggers/averaged_perceptron_tagger_eng/',
    'maxent_ne_chunker_tab': 'chunkers/maxent_ne_chunker_tab/',
    'words': 'corpora/words',
}
for package, resource in resources.items():
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(package, quiet=True, raise_on_error=True)

word_pattern = re.compile(r'\w+')
token_pattern = re.compile(r'\w+|[^\w\s]')
word_sets = {
    sentence: {word.casefold() for word in word_pattern.findall(sentence)}
    for sentence in unique_sentences
}

def words_and_entities(tree):
    items = []
    for node in tree:
        if isinstance(node, Tree):
            words = [word.casefold() for word, _ in node.leaves() if word_pattern.fullmatch(word)]
            if words:
                items.append(' '.join(words))
        else:
            word, _ = node
            if word_pattern.fullmatch(word):
                items.append(word.casefold())
    return items

In [3]:
example = 'John Smith is working.'
example_tree = nltk.ne_chunk(nltk.pos_tag(token_pattern.findall(example)), binary=True)
print(words_and_entities(example_tree))

tagged_sentences = nltk.pos_tag_sents([token_pattern.findall(sentence) for sentence in unique_sentences])
chunked_sentences = nltk.ne_chunk_sents(tagged_sentences, binary=True)
entity_sets = {
    sentence: set(words_and_entities(tree))
    for sentence, tree in zip(unique_sentences, chunked_sentences)
}

['john smith', 'is', 'working']


## Comparison

The gold scores measure similarity, so we use the Jaccard coefficient (one minus Jaccard distance). We compare both sets of scores with the gold scores using Pearson correlation.

In [4]:
from statistics import correlation

from nltk.metrics import jaccard_distance

def similarities(sets):
    return [
        1 - jaccard_distance(sets[a], sets[b]) if sets[a] or sets[b] else 1.0
        for a, b in pairs
    ]

word_scores = similarities(word_sets)
entity_scores = similarities(entity_sets)

print('Pair | Gold | Words | Words + NEs')
for i in range(10):
    print(f'{i + 1} | {gs[i]:.2f} | {word_scores[i]:.3f} | {entity_scores[i]:.3f}')

print(f'\nPearson (words): {correlation(gs, word_scores):.6f}')
print(f'Pearson (words + NEs): {correlation(gs, entity_scores):.6f}')
changed = sum(word_score != entity_score for word_score, entity_score in zip(word_scores, entity_scores))
print(f'Similarity changed for {changed} of {len(pairs)} pairs')

Pair | Gold | Words | Words + NEs
1 | 4.50 | 0.333 | 0.333
2 | 5.00 | 0.769 | 0.750
3 | 4.25 | 0.381 | 0.381
4 | 4.50 | 0.583 | 0.583
5 | 5.00 | 1.000 | 1.000
6 | 5.00 | 0.368 | 0.368
7 | 4.67 | 0.294 | 0.294
8 | 5.00 | 0.750 | 0.750
9 | 5.00 | 0.286 | 0.286
10 | 5.00 | 0.500 | 0.500

Pearson (words): 0.485346
Pearson (words + NEs): 0.488886
Similarity changed for 54 of 459 pairs


In [5]:
for pair_number in (57, 158):
    i = pair_number - 1
    print(f'Pair {pair_number} (gold: {gs[i]:.2f})')
    for sentence in pairs[i]:
        print(sentence)
        print('Multiword entities:', sorted(item for item in entity_sets[sentence] if ' ' in item))
    print(f'Words: {word_scores[i]:.3f}; words + NEs: {entity_scores[i]:.3f}\n')

Pair 57 (gold: 5.00)
Van Orden Report (A5-0241/2000)
Multiword entities: ['van orden report']
Van Orden report (A5-0241 / 2000)
Multiword entities: ['van orden']
Words: 1.000; words + NEs: 0.500

Pair 158 (gold: 3.00)
Van Orden Report (A5-0241/2000)
Multiword entities: ['van orden report']
Report Horsebox Orden (A5-0241 / 2000)
Multiword entities: ['report horsebox orden']
Words: 0.714; words + NEs: 0.600



## Justification

- Pearson 0.485 to 0.489 with NEs, smaill
- Pair 57, same report (gold 5/5). Chunker grabs "Van Orden Report" in one and "van orden" in the other. 1.000 to 0.500.
- Pair 158, different names share "report"  plus "Orden", 0.714 to 0.600.
- Useful when the full name matters, we should stay with words there, noo?